In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import h5py
import matplotlib.pyplot as plt
import numpy as np
import scipy.signal

import brighteyes_flim.tools_phasor as brighteyes_flim
import brighteyes_ism.analysis.Graph_lib as gr
import brighteyes_ism.dataio.mcs as mcs
from scipy.ndimage import shift
from scipy.optimize import curve_fit
from skimage.filters import gaussian
from skimage.registration import phase_cross_correlation


### Load the data and realign the temporal bins 

In [ ]:
data = h5py.File(r"C:\Users\REPLACE_ME\Desktop\images\Lamina_Tubulin\Lamina.h5", "r") 
print(data.keys())
image = data["data"]
image = np.sum(image, axis=(0,1), dtype=np.uint16)
print(image.shape)

In [ ]:
data_path_irf = r"\\iitfsvge101.iit.local\mms\Data MMS server\STED-ISM\AxialDeconvolution\flim\lamina+tubulin\data-10-04-2024-19-50-34.h5" 
data_irf = h5py.File(data_path_irf)

image_irf = data_irf["data"]
image_irf = np.sum(image_irf, axis=(0,1))
print(image_irf.shape)


In [ ]:
data_hist_irf = np.sum(image_irf, axis=(0, 1))
print('hist shape', data_hist_irf.shape)

In [ ]:
nch = data_hist_irf.shape[-1]
print("nchannels", nch)

shift_vec = np.empty( nch )

for i in range(nch):
    shift_vec[i], *_ = phase_cross_correlation(data_hist_irf[:, 12], data_hist_irf[:, i], upsample_factor=10, normalization=None)


irf_shifted = np.empty_like(data_hist_irf)
for i in range(nch):
    irf_shifted[:, i] = shift(data_hist_irf[:, i], shift_vec[i], order = 1, mode='grid-wrap')

In [ ]:
plt.figure()
for i in range(nch):
    plt.plot(irf_shifted[:,i])


In [ ]:
plt.figure()
hists = np.sum(image, axis=(0,1))
for i in range(nch):
    plt.plot(hists[:,i])

In [ ]:
nch = image.shape[-1]
image_4D = np.empty_like(image)
shift_dset = np.zeros( (image.ndim - 1, nch) )
shift_dset[-1] = shift_vec
print("shift_dset", shift_dset.shape)

for i in range(nch):
    image_4D[..., i] = shift(image[..., i], shift_dset[:, i], order = 1, mode='grid-wrap')
    
print("image_4D", image_4D.shape)



In [ ]:
hists_realigned = np.sum(image_4D, axis = (0,1))
plt.figure()
for i in range(nch):
    plt.plot(hists_realigned[:,i])

### APR reconstruction for spatial dimensions

In [ ]:
import brighteyes_ism.analysis.APR_lib as apr

print(image_4D.shape)
image_3d = np.sum(image_4D, axis = -2)
print(image_3d.shape)
shift_vectors, err = apr.ShiftVectors(image_3d, usf = 100, ref = 12)
gr.PlotShiftVectors(shift_vectors)
print (shift_vectors)     

In [ ]:
from tqdm import tqdm
with h5py.File(r"C:\Users\REPLACE_ME\Desktop\images\Lamina_Tubulin\APR_1_Lamina_and_Tubulin", 'w') as f:     
     x_size, y_size, bin_size, channel_size = image_4D.shape[0], image_4D.shape[1], image_4D.shape[2], image_4D.shape[3]
# Create an empty dataset with dimensions (x,y,t, ch)
     dataset_shape = (x_size, y_size, bin_size, channel_size)
     h5_dataset = f.create_dataset('data', shape=dataset_shape, dtype=np.uint16)
    

     

     for bin in tqdm(range(image_4D.shape[-2])):
         h5_dataset[:, :, bin, :] = apr.Reassignment(shift_vectors, image_4D[:, :, bin, :], mode = 'interp')

In [ ]:
f = h5py.File(r"C:\Users\REPLACE_ME\Desktop\images\Lamina_Tubulin\APR_1_Lamina_and_Tubulin", 'r') 
h5_dataset = f["data"]
print(h5_dataset.shape)
h5_dataset_sum = np.sum(h5_dataset, axis=-1)

In [ ]:
hists_realigned_apr = np.sum(h5_dataset, axis = (0,1))
plt.figure()
for i in range(nch):
    plt.plot(hists_realigned_apr[:,i])

### Plot intensity image (x,y) after performing APR and sum on the temporal bins

In [ ]:
data_histograms = np.sum(h5_dataset_sum, axis = -1)
print(data_histograms.shape)
    
# Plot the histogram of the photon counts in each pixel to see the distribution (e.g. check the level of noise) 
plt.figure()
plt.hist(data_histograms.flatten(), bins = 100, range = (0, 2000))

### Calculate and show the phasor plot of the 3D image without corrections

In [ ]:
phasor_matrix_1= brighteyes_flim.phasor(h5_dataset_sum)
print(phasor_matrix_1.shape)
brighteyes_flim.plot_phasor(phasor_matrix_1[:], bins_2dplot=200, log_scale=False, quadrant='all', dfd_freq = 41.48e6)

### Extract phasor of the IRF for correcting the measured phasors

In [ ]:
irf_summed = np.sum(irf_shifted, axis = -1)
plt.figure()
plt.plot(irf_summed)
print(irf_summed.shape)
phasor_ir = brighteyes_flim.phasor(irf_summed)
print(phasor_ir)

### Calculate the correction factor from the laser's (26th channel's) phasor to realign the phasors of the data in the phasor plot

In [ ]:
data_path = r"C:\Users\REPLACE_ME\Desktop\images\Lamina_Tubulin\Lamina.h5"   #change path
data_extra, _ = mcs.load(data_path, key="data_channels_extra")
data_laser = data_extra[:, :, :, :, :, 1]
print(data_laser.shape)
data_laser_hist = np.sum(data_laser, axis = (0,1,2,3))
phasor_laser = brighteyes_flim.calculate_phasor(data_laser_hist)
print(phasor_laser)

In [ ]:
data_extra_irf, _ = mcs.load(data_path_irf, key="data_channels_extra")
data_laser_irf = data_extra_irf[:, :, :, :, :, 1]
print(data_laser_irf.shape)
data_laser_hist_irf = np.sum(data_laser_irf, axis = (0,1,2,3))
phasor_laser_irf = brighteyes_flim.calculate_phasor(data_laser_hist_irf)
print(phasor_laser_irf)

In [ ]:
corr = brighteyes_flim.correction_phasor(data_laser_hist, data_laser_hist_irf)
print(corr)

### Show the phasor plot with the phasors corrected for the IRF and the laser phasor

In [ ]:
phasor_corrected = phasor_matrix_1 * corr / phasor_ir
brighteyes_flim.plot_phasor(phasor_corrected, bins_2dplot=200, log_scale=False, quadrant='first')

### Lifetime analysis with phasors 

In [ ]:
tau_phi = brighteyes_flim.calculate_tau_phi(np.real(phasor_corrected), np.imag(phasor_corrected))
print(tau_phi.shape)

In [ ]:
tau_m = brighteyes_flim.calculate_tau_m(np.real(phasor_corrected), np.imag(phasor_corrected))
print(tau_m.shape)

In [ ]:
tau_data = 1e9*tau_phi.flatten()

plt.figure()
plt.hist(tau_data, range = (-6, 6), bins = 50)

In [ ]:
tau_m_data = 1e9*tau_m.flatten()

plt.figure()
plt.hist(tau_m_data, range = (-6, 6), bins = 50)

In [ ]:
fig = plt.figure(figsize = (9, 6))
gs = fig.add_gridspec(4, 4)
ax1 = fig.add_subplot(gs[0:2, 0:2])
ax2 = fig.add_subplot(gs[2:4, 0:2])
brighteyes_flim.plot_phasor(phasor_corrected, bins_2dplot=200, log_scale=False, quadrant='first', fig = fig, ax = ax1)
gr.show_flim(data_histograms, tau_m*1e9, pxsize = 0.04, pxdwelltime = 162, lifetime_bounds = (2.2, 4.5), fig = fig, ax = ax2)  
fig.tight_layout()
plt.savefig(r"C:\Users\REPLACE_ME\Desktop\PDF_processed_images\Lamina_Tubulin\APR_tau_M_Lamina_and_Tubulin.pdf", dpi = 900)

In [ ]:
fig = plt.figure(figsize = (9, 6))
gs = fig.add_gridspec(4, 4)
ax1 = fig.add_subplot(gs[0:2, 0:2])
ax2 = fig.add_subplot(gs[2:4, 0:2])
brighteyes_flim.plot_phasor(phasor_corrected, bins_2dplot=200, log_scale=False, quadrant='first', fig = fig, ax = ax1)
gr.show_flim(data_histograms, tau_phi*1e9, pxsize = 0.04, pxdwelltime = 162, lifetime_bounds = (2.2, 4.5), fig = fig, ax = ax2)  
fig.tight_layout()
plt.savefig(r"C:\Users\REPLACE_ME\Desktop\PDF_processed_images\Lamina_Tubulin\APR_tau_Phi_Lamina_and_Tubulin.pdf", dpi = 900)

### Lifetime analysis with fitting

In [ ]:
hist_conv = np.sum(h5_dataset_sum, axis = (0,1))  #plot histogram 1D
plt.figure()
plt.plot(hist_conv)

### Test on global fitting

In [ ]:
def exp_fun(A, tau, t):
    decay = A * np.exp(-t/tau) 
   # decay[t<0] = 0
    return decay

def model_2(t, A, tau, bkg):
   
    dt = 0.297619 #ns
    nbin = 81
    period = dt * nbin
    
    irf = irf_summed.copy().astype(np.float64)                           ########## change the IRF with the actual IRF for this microscope
    irf/= irf.sum()
    decay = (np.heaviside(t, 0) + 1/(np.exp(period/tau) -1)) * exp_fun(A,tau,t)
    decay_convolved = scipy.signal.convolve(decay, irf, mode='same')
    decay_convolved =  decay_convolved[nbin//2:-nbin//2]
    return decay_convolved + bkg

def fit_function_2(t, A, tau, bkg):
    return model_2(t, A, tau, bkg)

In [ ]:
dt = 0.297619 #ns
nbin = 81                                                 #### update the parameters with the one of the DFD
period = dt * nbin
t=np.linspace(-period, period, nbin*2)
t_1 = np.linspace(0, period, nbin) 

decay = (np.heaviside(t, 0) + 1/(np.exp(period/2.5) -1)) * exp_fun(10,2.5,t)
decay_convolved = scipy.signal.convolve(decay, irf_summed, mode='same')
decay_convolved =  decay_convolved[nbin//2:-nbin//2]

plt.figure()
plt.plot(t_1, decay_convolved)

In [ ]:
print(np.argmax(decay_convolved))
print(np.argmax(irf_summed))
print(np.argmax(hist_conv))

In [ ]:
phot_max = np.max(hist_conv)
print(np.max(hist_conv))

In [ ]:
phot_sum = np.sum(hist_conv)
print(phot_sum)

In [ ]:
hist_shifted = shift(hist_conv, -2, order=1, mode='grid-wrap')     ### change the shift value in case
print(np.argmax(hist_shifted))

In [ ]:
hist_shifted_1 = shift(h5_dataset_sum[270,220,:], -2, order=1, mode='grid-wrap')
print(np.argmax(hist_shifted_1))

In [ ]:
dt = 0.297619 #ns                           ### change the parameters
nbin = 81 
per = dt*nbin
t_axis = np.linspace(-per, per, nbin*2)
initial_guess = [650000, 2.5, 4000]

# Perform the fit
popt, pcov = scipy.optimize.curve_fit(fit_function_2, t_axis, hist_shifted_1, p0=initial_guess)

# Extract fitted parameters
fitted_A, fitted_tau, fitted_bkg  = popt

In [ ]:
print(f"Fitted parameters: A = {fitted_A}, tau = {fitted_tau}, Background noise = {fitted_bkg}")

In [ ]:
t = np.linspace(0, 24.1, 81)
plt.figure(figsize=(10, 6))
plt.plot(t, hist_shifted_1, 'b-', label='measured data')
plt.plot(t, fit_function_2(t_axis, *popt), 'r--', label='Fitted model')
plt.xlabel('Time')
plt.ylabel('Intensity')
plt.legend()
plt.show()

### reconstruct the lifetime in each pixel

In [ ]:
lifetime_map = np.zeros((h5_dataset_sum.shape[0], h5_dataset_sum.shape[1]))
background_map = np.zeros((h5_dataset_sum.shape[0], h5_dataset_sum.shape[1]))
initial_guess = [500, 2.5, 50]
dt = 0.297619 #ns
nbin = 81
per = dt*nbin
t_axis = np.linspace(-per, per, nbin*2)

for i in range(h5_dataset_sum.shape[0]):
    for j in range(h5_dataset_sum.shape[1]):
       
        decay_histogram = h5_dataset_sum[i, j, :]
        decay_histogram = shift(decay_histogram, -2, order=1, mode='grid-wrap')       ############### change the shift term
        
        try:
            popt, pcov = scipy.optimize.curve_fit(fit_function_2, t_axis, decay_histogram, p0=initial_guess)
            fitted_tau = popt[1]
            
        
        except RuntimeError:
            fitted_tau = np.nan
        
        
        lifetime_map[i, j] = fitted_tau
       

In [ ]:
plt.figure()
lifetime_flattened = lifetime_map.flatten()
plt.hist(lifetime_flattened, range = (-3, 8), bins = 50)

In [ ]:
fig = plt.figure(figsize = (9, 6))
gr.show_flim(data_histograms, lifetime_map, pxsize = 0.04, pxdwelltime = 162, lifetime_bounds = (2.2, 4.5), fig = fig)  
fig.tight_layout()
plt.savefig(r"C:\Users\REPLACE_ME\Desktop\PDF_processed_images\Lamina_Tubulin\Fitting_APR_Lamina_Tubulin.pdf", dpi = 900)

## Confocal analysis

### Phasor analysis on confocal-like image

In [ ]:
data_3D = image_4D.sum(-1)

In [ ]:
data_histograms_confocal = np.sum(data_3D, axis = -1)
print(data_histograms_confocal.shape)
    
# Plot the histogram of the photon counts in each pixel to see the distribution (e.g. check the level of noise) 
plt.figure()
plt.hist(data_histograms_confocal.flatten(), bins = 100, range = (0, 2000))

In [ ]:
phasor_matrix_confocal= brighteyes_flim.phasor(data_3D)
print(phasor_matrix_confocal.shape)
brighteyes_flim.plot_phasor(phasor_matrix_confocal[:], bins_2dplot=200, log_scale=False, quadrant='all', dfd_freq = 41.48e6)

In [ ]:
phasor_clean = phasor_matrix_confocal * corr / phasor_ir
brighteyes_flim.plot_phasor(phasor_clean, bins_2dplot=200, log_scale=False, quadrant='first')

In [ ]:
tau_phi_confocal = brighteyes_flim.calculate_tau_phi(np.real(phasor_clean), np.imag(phasor_clean))
print(tau_phi_confocal.shape)

In [ ]:
tau_m_confocal = brighteyes_flim.calculate_tau_m(np.real(phasor_clean), np.imag(phasor_clean))
print(tau_m_confocal.shape)

In [ ]:
tau_data_confocal = 1e9*tau_phi_confocal.flatten()

plt.figure()
plt.hist(tau_data_confocal, range = (-6, 6), bins = 50)

In [ ]:
tau_m_data_confocal = 1e9*tau_m_confocal.flatten()

plt.figure()
plt.hist(tau_m_data_confocal, range = (-6, 6), bins = 50)

In [ ]:
fig = plt.figure(figsize = (9, 6))
gs = fig.add_gridspec(4, 4)
ax1 = fig.add_subplot(gs[0:2, 0:2])
ax2 = fig.add_subplot(gs[2:4, 0:2])
brighteyes_flim.plot_phasor(phasor_clean, bins_2dplot=200, log_scale=False, quadrant='first', fig = fig, ax = ax1)
gr.show_flim(data_histograms_confocal, tau_m_confocal*1e9, pxsize = 0.04, pxdwelltime = 162, lifetime_bounds = (2.0, 4.5), fig = fig, ax = ax2)  
fig.tight_layout()
plt.savefig(r"C:\Users\REPLACE_ME\Desktop\PDF_processed_images\Lamina_Tubulin\Confocal_tau_M_Lamina_and_Tubulin.pdf", dpi = 900)

In [ ]:
fig = plt.figure(figsize = (9, 6))
gs = fig.add_gridspec(4, 4)
ax1 = fig.add_subplot(gs[0:2, 0:2])
ax2 = fig.add_subplot(gs[2:4, 0:2])
brighteyes_flim.plot_phasor(phasor_clean, bins_2dplot=200, log_scale=False, quadrant='first', fig = fig, ax = ax1)
gr.show_flim(data_histograms_confocal, tau_phi_confocal*1e9, pxsize = 0.04, pxdwelltime = 162, lifetime_bounds = (2.0, 4.5), fig = fig, ax = ax2)  
fig.tight_layout()
plt.savefig(r"C:\Users\REPLACE_ME\Desktop\PDF_processed_images\Lamina_Tubulin\Confocal_tau_Phi_Lamina_and_Tubulin.pdf", dpi = 900)

### Lifetime analysis on Confocal-like image

In [ ]:
hist_conv_conf = np.sum(data_3D, axis = (0,1))  #plot histogram 1D
plt.figure()
plt.plot(hist_conv_conf)

In [ ]:
print(np.argmax(decay_convolved))
print(np.argmax(irf_summed))
print(np.argmax(hist_conv_conf))

In [ ]:
hist_shifted_conf = shift(hist_conv_conf, -2, order=1, mode='grid-wrap')     ### change the shift value in case
print(np.argmax(hist_shifted_conf))

In [ ]:
dt = 0.297619 #ns                           ### change the parameters
nbin = 81 
per = dt*nbin
t_axis = np.linspace(-per, per, nbin*2)
initial_guess_c = [65000, 2.5, 4000]


# Perform the fit
popt_conf, pcov_conf = scipy.optimize.curve_fit(fit_function_2, t_axis, hist_shifted_conf, p0=initial_guess_c)

# Extract fitted parameters
fitted_A_conf, fitted_tau_conf, fitted_bkg_conf  = popt_conf

In [ ]:
print(f"Fitted parameters: A = {fitted_A_conf}, tau = {fitted_tau_conf}, Background noise = {fitted_bkg_conf}")

In [ ]:
t = np.linspace(0, 24.1, 81)
plt.figure(figsize=(10, 6))
plt.plot(t, hist_shifted_conf, 'b-', label='measured data')
plt.plot(t, fit_function_2(t_axis, *popt_conf), 'r--', label='Fitted model')
plt.xlabel('Time')
plt.ylabel('Intensity')
plt.legend()
plt.show()

In [ ]:
lifetime_map_conf = np.zeros((data_3D.shape[0], data_3D.shape[1]))
background_map_conf = np.zeros((data_3D.shape[0], data_3D.shape[1]))
initial_guess_c = [500, 2.5, 50]
dt = 0.297619 #ns
nbin = 81
per = dt*nbin
t_axis = np.linspace(-per, per, nbin*2)

for i in range(data_3D.shape[0]):
    for j in range(data_3D.shape[1]):
       
        decay_histogram_c = data_3D[i, j, :]
        decay_histogram_c = shift(decay_histogram_c, -2, order=1, mode='grid-wrap')       ############### change the shift term
        
        try:
            popt_c, pcov_c = scipy.optimize.curve_fit(fit_function_2, t_axis, decay_histogram_c, p0=initial_guess_c)
            fitted_tau_c = popt_c[1]
            
        
        except RuntimeError:
            fitted_tau_c = np.nan
        
        
        lifetime_map_conf[i, j] = fitted_tau_c

In [ ]:
plt.figure()
lifetime_flattened_c = lifetime_map_conf.flatten()
plt.hist(lifetime_flattened_c, range = (-3, 8), bins = 50)

In [ ]:
fig = plt.figure(figsize = (9, 6))
gr.show_flim(data_histograms_confocal, lifetime_map_conf, pxsize = 0.04, pxdwelltime = 162, lifetime_bounds = (2.0, 4.5), fig = fig)  
fig.tight_layout()
plt.savefig(r"C:\Users\REPLACE_ME\Desktop\PDF_processed_images\Lamina_Tubulin\Fitting_Confocal_Lamina_Tubulin.pdf", dpi = 900)

### Phasor segmentation

In [ ]:
import brighteyes_flim.tools_phasor as flim
thresholded_phasor_sum = flim.threshold_phasor(data_histograms, phasor_corrected, 0.05)
flim.plot_phasor(phasor_corrected, bins_2dplot=200, log_scale=False, quadrant='first')


In [ ]:
print(thresholded_phasor_sum.shape)
print(phasor_corrected.shape)

In [ ]:
from sklearn.mixture import GaussianMixture
import matplotlib as mpl

from numpy.linalg import inv

colors = ["navy", "darkorange"]

def make_ellipses(gmm, ax):
    for n, color in enumerate(colors):
        if gmm.covariance_type == "full":
            covariances = gmm.covariances_[n][:2, :2]
        elif gmm.covariance_type == "tied":
            covariances = gmm.covariances_[:2, :2]
        elif gmm.covariance_type == "diag":
            covariances = np.diag(gmm.covariances_[n][:2])
        elif gmm.covariance_type == "spherical":
            covariances = np.eye(gmm.means_.shape[1]) * gmm.covariances_[n]
        v, w = np.linalg.eigh(covariances)
        u = w[0] / np.linalg.norm(w[0])
        angle = np.arctan2(u[1], u[0])
        angle = 180 * angle / np.pi  # convert to degrees
        v = 2.0 * np.sqrt(2.0) * np.sqrt(v)
        ell = mpl.patches.Ellipse(
            gmm.means_[n, :2], v[0], v[1], angle=180 + angle, color=color
        )
        ell.set_clip_box(ax.bbox)
        ell.set_alpha(0.5)
        ax.add_artist(ell)
        ax.set_aspect("equal", "datalim")

def segment_phasors(gmm, phasor_map, distance = 'mahalanobis'):
    
    sz = phasor_map.shape
    
    phasor = np.asarray([np.real(phasor_map), np.imag(phasor_map)])
    means = gmm.means_
    if distance == 'mahalanobis':
        cov = gmm_sum.covariances_
        cov_inv = np.asarray([inv(cov[n]) for n in range(cov.shape[0])]) # mahalanobis distance
    elif distance == 'euclid':
        cov_inv = np.asarray([np.eye(2) for n in range(cov.shape[0])]) # euclid distance
    
    n_components = gmm.get_params()['n_components']
    distance = np.empty( (n_components,) + sz )
    for n in range(n_components):
        # distance[n] = np.sqrt( (phasor[0] - means[n,0])**2 + (phasor[1] - means[n,1])**2 )
        
        v = np.asarray( [phasor[0] - means[n,0], phasor[1] - means[n,1] ] )
        w = np.einsum('ixy, ij, jxy -> xy', v, cov_inv[n], v)
        distance[n] = np.sqrt( w )
        
    segment = np.argmin(distance, axis = 0)
        
    mask = np.zeros( (n_components,) + phasor_map.shape )
    for n in range(n_components):
        mask[n][segment == n] = 1
    
    return mask, segment


def draw_decision_boundary(gmm, ax, color = 'red'):
    
    x = np.arange(0, 1, 0.001)
    y = np.arange(0, 0.6, 0.001)
    xx, yy = np. meshgrid(x, y)

    zz = xx + 1j*yy

    mask, segment = segment_phasors(gmm, zz)
    levels = np.arange(segment.min(), segment.max())
    
    ax.scatter(gmm.means_[:,0], gmm.means_[:,1], color = color, s = 20)
    ax.contour(xx, yy, segment, levels = levels, colors = color)

In [ ]:
img = np.asarray([np.real(thresholded_phasor_sum), np.imag(thresholded_phasor_sum)]).T.copy()

classif = GaussianMixture(n_components=2)
gmm_sum = classif.fit(img)

fig, ax = flim.plot_phasor(thresholded_phasor_sum, quadrant='first', bins_2dplot = 200, cmap='viridis')
make_ellipses(gmm_sum, ax)
ax.scatter(gmm_sum.means_[:,0], gmm_sum.means_[:,1], color = 'red', s = 20)

mask_sum, _ = segment_phasors(gmm_sum, phasor_corrected)
mask_sum = np.flip(mask_sum, axis = 0)

#%%

fig, ax = flim.plot_phasor(thresholded_phasor_sum, quadrant='first', bins_2dplot = 200, cmap='viridis')
draw_decision_boundary(gmm_sum, ax)

#%%



In [ ]:
print(gmm_sum.means_[:,0])
print(gmm_sum.means_[:,1])

In [ ]:
print(mask_sum.shape)
print(img.shape)

In [ ]:
import os
path = r'\\iitfsvge101.iit.local\mms\Data MMS server\STED-ISM\AxialDeconvolution\flim\lamina+tubulin'

file = r'data-10-04-2024-19-27-47.h5'

fullpath = os.path.join(path, file)

data, meta = mcs.load(fullpath)

In [ ]:

crop = 30
fig, ax = plt.subplots(2, 4, figsize = (20, 10))

gr.ShowImg(data_histograms[crop:-crop, crop:-crop], meta.dx, meta.pxdwelltime, fig = fig, ax = ax[0, 0])
for n in range(2):
    gr.ShowImg(data_histograms[crop:-crop, crop:-crop]*mask_sum[n][crop:-crop, crop:-crop], meta.dx, meta.pxdwelltime, fig = fig, ax = ax[0, n+1])
    
flim.plot_phasor(thresholded_phasor_sum, quadrant='first', bins_2dplot = 200, cmap='viridis', fig = fig, ax = ax[0, 3])
draw_decision_boundary(gmm_sum, ax[0, 3])

In [ ]:
fig = plt.figure(figsize = (9, 6))
gs = fig.add_gridspec(4, 4)
ax1 = fig.add_subplot(gs[0:2, 0:2])
ax2 = fig.add_subplot(gs[2:4, 0:2])
gr.ShowImg(data_histograms[crop:-crop, crop:-crop]*mask_sum[0][crop:-crop, crop:-crop], meta.dx, meta.pxdwelltime, fig = fig, ax = ax1)
gr.ShowImg(data_histograms[crop:-crop, crop:-crop]*mask_sum[1][crop:-crop, crop:-crop], meta.dx, meta.pxdwelltime, fig = fig, ax = ax2)
fig.tight_layout()

In [ ]:
for n in range(2):
    gr.ShowImg(mask_sum[n], meta.dx, meta.pxdwelltime, fig = fig, ax = ax[0, n])
    
    
#%%

from scipy.stats import entropy

print( entropy(mask_sum[1]).sum() )



In [ ]:
print(M_inv.shape)

### Using phasors to split the concentrations of the 2 fluorophores in each pixel

### Method 1: Solve the linear system without constraints

In [ ]:

M = np.array([[0.7, 0.50],         ## Matrix with phasors of single species 
              [0.43, 0.48]]) 

# Compute the inverse of M
M_inv = np.linalg.inv(M)           # (g,s) =(2,2)

P = np.asarray((np.real(phasor_corrected), np.imag(phasor_corrected))) 
P=np.swapaxes(P, 0, -1)   #(x,y,j) =(1250,1250,2)

#P1 = P[440,230,:]
# Use einsum to perform the matrix multiplication for each pixel
# This will multiply M_inv by the phasor at each pixel
f = np.einsum('gs,xys->xyg', M_inv, P)


f1 = f[:, :, 0]  
f2 = f[:, :, 1]  



### Method 2: Solve the linear system with constraints

In [ ]:
M = np.array([[0.7, 0.50],         ## Matrix with phasors of single species 
              [0.43, 0.48]])

In [ ]:
from tqdm import trange
from scipy.optimize import minimize

def loss(x, m, p):
    return np.linalg.norm( m @ x - p )**2
    
def split(m, p):
    
    func = lambda x : loss(x, m, p)

    bounds = ((0, 1), (0,1))
    constraints = ({'type': 'eq', 'fun': lambda x:  x[0] + x[1] - 1})

    x0 = np.asarray( [0.5, 0.5] )

    res = minimize(func, x0, method='SLSQP',
                   bounds=bounds, constraints=constraints)
    
    return res.x


def image_split(phasor_map, matrix):
    
    sz = phasor_map.shape
    n_components = matrix.shape[1]
    
    phasor_tensor = np.asarray( (np.real(phasor_map), np.imag(phasor_map)) )
    phasor_tensor = np.moveaxis(phasor_tensor, 0, -1)
    
    weights = np.empty( sz + (n_components,) )
    
    num_pixels = np.prod(sz)
    
    for n in trange(num_pixels):
        idx = np.unravel_index(n, sz)
        weights[idx] = split(matrix, phasor_tensor[idx])
        
    return weights

In [ ]:
weights_f1_f2 = image_split(phasor_corrected, M)

In [ ]:
print(weights_f1_f2.shape)

In [ ]:
w1 = weights_f1_f2[:, :, 0]
w2 = weights_f1_f2[:, :, 1]

In [ ]:
print(w1.shape)
print(w2.shape)

In [ ]:
fig = plt.figure(figsize = (9, 6))
gs = fig.add_gridspec(4, 4)
ax1 = fig.add_subplot(gs[0:2, 0:2])
ax2 = fig.add_subplot(gs[2:4, 0:2])
gr.ShowImg(data_histograms*w1, meta.dx, meta.pxdwelltime, fig = fig, ax = ax1)
gr.ShowImg(data_histograms*w2, meta.dx, meta.pxdwelltime, fig = fig, ax = ax2)
fig.tight_layout()

In [ ]:
fig = plt.figure(figsize = (9, 6))
gs = fig.add_gridspec(4, 4)
ax1 = fig.add_subplot(gs[0:2, 0:2])
ax2 = fig.add_subplot(gs[2:4, 0:2])
gr.ShowImg(data_histograms, meta.dx, meta.pxdwelltime, fig = fig, ax = ax1)
gr.ShowImg(data_histograms, meta.dx, meta.pxdwelltime, fig = fig, ax = ax2)
fig.tight_layout()

In [ ]:
print(f1.shape)
print(f2.shape)

In [ ]:
plt.figure()
plt.hist(f1.flatten(), range = (-1, 3), bins = 50)

In [ ]:
plt.figure()
plt.hist(f2.flatten(), range = (-1, 3), bins = 50)

In [ ]:
f1 = np.clip(f1, 0, 1)
f2 = np.clip(f2, 0, 1)

In [ ]:
plt.figure()
plt.hist(f2.flatten(), range = (-1, 3), bins = 50)

In [ ]:
plt.figure()
plt.hist(f1.flatten(), range = (-1, 3), bins = 50)

In [ ]:
fig = plt.figure(figsize = (9, 6))
gs = fig.add_gridspec(4, 4)
ax1 = fig.add_subplot(gs[0:2, 0:2])
ax2 = fig.add_subplot(gs[2:4, 0:2])
gr.ShowImg(data_histograms*f1.T, meta.dx, meta.pxdwelltime, fig = fig, ax = ax1)
gr.ShowImg(data_histograms*f2.T, meta.dx, meta.pxdwelltime, fig = fig, ax = ax2)
fig.tight_layout()

In [ ]:
plt.imshow(data_histograms*f1.T)

In [ ]:
plt.imshow(data_histograms * f2.T)

In [ ]:
fig1 = plt.figure(figsize = (9, 6))
gr.ShowImg(data_histograms, meta.dx, meta.pxdwelltime, fig = fig1)

In [ ]:
crop = 30
fig, ax = plt.subplots(2, 4, figsize = (20, 10))

gr.ShowImg(data_histograms[crop:-crop, crop:-crop], meta.dx, meta.pxdwelltime, fig = fig, ax = ax[0, 0])
for n in range(2):
    gr.ShowImg(data_histograms[crop:-crop, crop:-crop]*mask_sum[n][crop:-crop, crop:-crop], meta.dx, meta.pxdwelltime, fig = fig, ax = ax[0, n+1])
    
flim.plot_phasor(thresholded_phasor_sum, quadrant='first', bins_2dplot = 200, cmap='viridis', fig = fig, ax = ax[0, 3])
draw_decision_boundary(gmm_sum, ax[0, 3])


### Using fitting to split the concentrations of the 2 fluorophores in each pixel

In [ ]:
def exp_decay(t, tau):
    return np.exp(-t / tau)

# Bi-exponential model
def model_biexponential(t, A1, A2, tau1, tau2, irf, dt=0.297619, nbin=81):
    period = dt * nbin
    # Apply the IRF convolution
    decay1 = (np.heaviside(t, 0) + 1/(np.exp(period/tau1) - 1)) * exp_decay(t, tau1)
    decay2 = (np.heaviside(t, 0) + 1/(np.exp(period/tau2) - 1)) * exp_decay(t, tau2)
    irf = irf/np.sum(irf)
    #decay1 = exp_decay(t, A1, tau1)
    #decay2 = exp_decay(t, A2, tau2)
    #total_decay = decay1  + decay2 
    #decay_convolved = scipy.signal.convolve(total_decay, irf, mode='same')
    #decay_convolved = decay_convolved/np.max(decay_convolved)  
    norm_decay1 = np.zeros_like(decay1)
    np.divide(decay1, np.sum(decay1[nbin//2:-nbin//2]), out=norm_decay1, where=np.sum(decay1[nbin//2:-nbin//2])>0 )
    norm_decay2 = np.zeros_like(decay2)
    np.divide(decay2, np.sum(decay2[nbin//2:-nbin//2]), out=norm_decay2, where=np.sum(decay2[nbin//2:-nbin//2])>0 )
    total_decay = A1 * norm_decay1 + A2 * norm_decay2
    decay_convolved = scipy.signal.convolve(total_decay, irf, mode='same')
    decay_convolved = decay_convolved[nbin//2:-nbin//2]
    
    return decay_convolved

def poisson_log_likelihood(model_decay, observed_decay):
    return np.sum(model_decay - observed_decay * np.log(model_decay + 1e-10))  # Add small epsilon to avoid log(0)


# Loss function: compares the observed decay to the model decay
#def loss(weights, t, observed_decay, tau1, tau2, irf):
 #   A1, A2 = weights
  #  model_decay = model_biexponential(t, A1, A2, tau1, tau2, irf)
    
    # Calculate the squared error
   # return np.linalg.norm(model_decay - observed_decay)**2

def loss(weights, t, observed_decay, tau1, tau2, irf):
    A1, A2 = weights
    model_decay = model_biexponential(t, A1, A2, tau1, tau2, irf)
    
    # Poisson log-likelihood loss
    return poisson_log_likelihood(model_decay, observed_decay)

# Function to estimate fluorophore weights in one pixel
def estimate_weights(t, observed_decay, tau1, tau2, irf):
    initial_guess = [0.50, 0.50]
    bounds = [(0, 1), (0, 1)]
    constraints = {'type': 'eq', 'fun': lambda x: x[0] + x[1] - 1}
    
    result = minimize(loss, initial_guess, args=(t, observed_decay, tau1, tau2, irf),
                      method='SLSQP', bounds=bounds, constraints=constraints) # 
    
    return result.x

# Function to apply the fluorophore separation to an image
def fluorophore_unmixing(time_points, decay_map, tau1, tau2, irf):
    image_size_x, image_size_y, time_points_len = decay_map.shape
    weights_map = np.zeros((image_size_x, image_size_y, 2))  # For storing A1 and A2
    
    for x in range(image_size_x):
        for y in range(image_size_y):
            decay_curve = decay_map[x, y, :]
            weights_map[x, y, :] = estimate_weights(time_points, decay_curve, tau1, tau2, irf)
    
    return weights_map



In [ ]:
decay1 = (np.heaviside(t, 0) + 1/(np.exp(24.1/3.5) - 1)) * exp_decay(t, 3.5)
decay2 = (np.heaviside(t, 0) + 1/(np.exp(24.1/3.0) - 1)) * exp_decay(t, 3.0)
nbin=81
n1=np.sum(decay1[nbin//2:-nbin//2])
n2=np.sum(decay2[nbin//2:-nbin//2])
plt.plot(decay1[nbin//2:-nbin//2] )
plt.plot(decay1)
plt.plot(decay2)
print(n1)
print(n2)
print(np.sum(decay1))
print(np.sum(decay1[nbin//2:-nbin//2]/n1))
print(np.sum(decay2[nbin//2:-nbin//2]/n2))
print(np.sum(0.5*decay1[nbin//2:-nbin//2]/n1+0.5*decay2[nbin//2:-nbin//2]/n2))

### Tests on synthetic data

In [ ]:
def generate_synthetic_data(image_size, tau1, tau2, irf, t, bkg=0):
    # Create arrays for A1 and A2 (random concentrations of the two fluorophores)
    A1 = np.random.uniform(0, 1, size=(image_size, image_size))
    A2 = 1 - A1  
    
    
    I_measured = np.zeros((image_size, image_size, 81))
    
    for x in range(image_size):
        for y in range(image_size):
            decay1 = A1[x, y] * np.exp(-t/tau1)
            decay2 = A2[x, y] * np.exp(-t/tau2)
            
            # Total decay is the sum of both decays, convolved with the IRF
            decay_total = decay1 + decay2
            decay_convolved = scipy.signal.convolve(decay_total, irf, mode='same')
            #decay_convolved = decay_convolved[81//2:-81//2]
            
            # Add background noise
            I_measured[x, y, :] = decay_convolved + bkg
    
    return A1, A2, I_measured

In [ ]:
image_size = 50  
t_s = np.linspace(0, 24.1, 81)  
tau1 = 2.0  
tau2 = 3.0 

A1_true, A2_true, I_simulated = generate_synthetic_data(image_size, tau1, tau2, irf, t_s)

In [ ]:

plt.figure()
plt.plot(t_s, I_simulated[0,20,:])

In [ ]:
print(np.argmax(I_simulated[0,20,:]))

In [ ]:
A_pred = fluorophore_unmixing(t_s, I_simulated, tau1=2.0, tau2=3.0, irf=irf) 

In [ ]:
print(A_pred[:, :, 0] - A1_true)


In [ ]:
print(A_pred[:,:,1] - A2_true)

In [ ]:
plt.figure()
plt.hist(A1_true.flatten(), range = (0, 1), bins = 5)

In [ ]:
plt.figure()
plt.hist(A_pred[:,:,0].flatten(), range = (0, 1), bins = 5)

In [ ]:
plt.figure()
plt.hist(A_pred[:,:,1].flatten(), range = (0, 1), bins = 5)

### Real data

In [ ]:
model = model_biexponential(t, A1=0.3, A2=0.7, tau1=3.0, tau2=3.5, irf=irf_summed)
plt.figure()
plt.plot(t_s, model)

In [ ]:
np.argmax(model)
np.argmax(h5_dataset_sum[70,70, :])

In [ ]:
data_hope = shift(h5_dataset_sum, -2, order=1, mode='grid-wrap')

In [ ]:
data_nor = np.zeros((data_hope.shape[0], data_hope.shape[1], data_hope.shape[-1]))
for x in range(data_hope.shape[0]):
    for y in range(data_hope.shape[1]):
        data_nor[x,y,:] = data_hope[x,y,:]/np.sum(data_hope[x,y,:])

In [ ]:
data_h5_nor = np.zeros((h5_dataset_sum[270:470, 0:200,:].shape[0], h5_dataset_sum[270:470, 0:200,:].shape[1], h5_dataset_sum[270:470, 0:200,:].shape[-1]))
for x in range(h5_dataset_sum[270:470, 0:200,:].shape[0]):
    for y in range(h5_dataset_sum[270:470, 0:200,:].shape[1]):
        data_h5_nor[x,y,:] = h5_dataset_sum[270:470, 0:200,:][x,y,:]/np.sum(h5_dataset_sum[270:470, 0:200,:][x,y,:])

In [ ]:
plt.figure()
plt.plot(t_s, data_nor[0,0,:])
plt.plot(t_s, model, )
print(np.max(data_nor[0,0,:]))

In [ ]:
print(data_nor.shape)
print(t.shape)

In [ ]:
pesi = fluorophore_unmixing(t, data_nor, tau1=3.0, tau2=3.5, irf=irf_summed)

In [ ]:
pesi_a1 = pesi[:,:,0]
pesi_a2 = pesi[:,:,1]

In [ ]:
print(pesi_a1.shape)

In [ ]:
print(pesi_a1.max())
print(pesi_a2.max())
print(pesi_a1.min())
print(pesi_a2.min())

In [ ]:
print(np.mean(pesi_a1))
print(np.mean(pesi_a2))

In [ ]:
plt.figure()
plt.hist(pesi_a1.flatten(), range = (0, 1), bins = 10)

In [ ]:
plt.figure()
plt.hist(pesi_a2.flatten(), range = (0, 1), bins = 10)

In [ ]:
fig = plt.figure(figsize = (9, 6))
gs = fig.add_gridspec(4, 4)
ax1 = fig.add_subplot(gs[0:2, 0:2])
ax2 = fig.add_subplot(gs[2:4, 0:2])
gr.ShowImg(data_histograms * pesi_a1, meta.dx, meta.pxdwelltime, fig = fig, ax = ax1)
gr.ShowImg(data_histograms * pesi_a2, meta.dx, meta.pxdwelltime, fig = fig, ax = ax2)
fig.tight_layout()

In [ ]:
fig = plt.figure(figsize = (9, 6))
gr.ShowImg(data_histograms, meta.dx, meta.pxdwelltime, fig = fig)
plt.show

In [ ]:
model1 =  model_biexponential(t, A1=pesi_a1[0,0], A2=pesi_a2[0,0], tau1=3.0, tau2=3.5, irf=irf_summed)
#model2 =  model_biexponential(t, A1=1, A2=0, tau1=3.0, tau2=3.5, irf=irf_summed)
#model3 =  model_biexponential(t, A1=0, A2=1, tau1=3.0, tau2=3.5, irf=irf_summed)
plt.figure()
plt.plot(t_s, data_nor[0,0,:], label = 'dato') #/np.sum(data_nor[0,0,:]
plt.plot(t_s, model1, label= 'model1')
#plt.plot(t_s, model2, label= 'model2')
#plt.plot(t_s, model3, label= 'model3')
plt.legend()

In [ ]:
print(np.sum(model1))
print(np.sum(model2))
print(np.sum(model3))
print(np.sum(data_nor[0,0,:]))